In [6]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

In [7]:

np.random.seed(42)
data_dir = Path.cwd().parent /"data"/"processed"/"submissions"
train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")
sample = pd.read_csv(data_dir/"submission.csv")


In [8]:

feat_cols = [c for c in train.columns if c not in ["id", "anomaly"]]
for df in [train, test]:
    df["channel"] = df["channel"].astype("category")
all_channels = pd.concat([train["channel"], test["channel"]]).astype("category").cat.categories
train["channel"] = train["channel"].cat.set_categories(all_channels)
test["channel"] = test["channel"].cat.set_categories(all_channels)

X, y = train[feat_cols], train["anomaly"]
X_test = test[feat_cols]
X_onehot = pd.get_dummies(X, columns=["channel"], drop_first=True)
X_test_onehot = pd.get_dummies(X_test, columns=["channel"], drop_first=True).reindex(columns=X_onehot.columns, fill_value=0)
cat_idx = [X.columns.get_loc("channel")]

In [9]:

def save_submission(preds, name):
    sub = test[["id"]].copy()
    sub["anomaly"] = preds
    sub = sample[["id"]].merge(sub, on="id", how="left")
    assert sub.shape[0] == 425 and sub["anomaly"].isna().sum() == 0
    sub.to_csv(data_dir / name, index=False)
    print(f"{name}: predicted anomaly rate = {sub.anomaly.mean():.4f}")


In [10]:
# --- A) LightGBM original ---
lgb_model = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.03, num_leaves=15,
    min_child_samples=15, subsample=0.8, colsample_bytree=0.8,
    class_weight="balanced", random_state=42, verbosity=-1,
)
lgb_model.fit(X, y, categorical_feature=["channel"])
lgb_probs = lgb_model.predict_proba(X_test)[:, 1]
save_submission((lgb_probs > 0.46).astype(int), "submission_A_lightgbm.csv")

submission_A_lightgbm.csv: predicted anomaly rate = 0.2141


In [11]:
# --- B) CatBoost ---
cb_model = CatBoostClassifier(
    iterations=400, depth=5, learning_rate=0.05,
    l2_leaf_reg=5, auto_class_weights="Balanced",
    cat_features=cat_idx, random_state=42, verbose=False,
)
cb_model.fit(X, y)
cb_probs = cb_model.predict_proba(X_test)[:, 1]
save_submission((cb_probs > 0.51).astype(int), "submission_B_catboost.csv")

submission_B_catboost.csv: predicted anomaly rate = 0.2071


In [12]:
# --- C) Blend ---
rf_model = RandomForestClassifier(n_estimators=400, max_depth=6, min_samples_leaf=5,
                                   class_weight="balanced", random_state=42, n_jobs=-1)
rf_model.fit(X_onehot, y)
rf_probs = rf_model.predict_proba(X_test_onehot)[:, 1]

blend_probs = (lgb_probs + cb_probs + rf_probs) / 3
save_submission((blend_probs > 0.50).astype(int), "submission_C_blend.csv")

submission_C_blend.csv: predicted anomaly rate = 0.2071
